# 02 · Preprocessing for feature-based models

<a href="https://colab.research.google.com/github/dokunoale/chagas/blob/refactor/notebooks/02_preprocessing_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook builds the cohort shared by both pipelines (quality checks, deduplication, patient-level folds) and the inputs of the feature-based models: band-passed signals at 100 Hz and physiological features. The choices follow from `01_dataset_analysis.ipynb`. The spectrogram pipeline for CNNs is in `03_preprocessing_dl.ipynb` and reuses the cohort saved here.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/chagas")
    if ROOT.exists():
        !git -C {ROOT} pull -q
    else:
        !git clone -q -b refactor https://github.com/dokunoale/chagas.git {ROOT}
    %pip install -q neurokit2
else:
    ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(ROOT))

In [ ]:
import warnings
from math import ceil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from src import ecg, features, leakage
from src import preprocessing as pp
from src.dataset import download, sources
from src.metrics import challenge_score

DATA_DIR = Path("/content/data") if IN_COLAB else ROOT / "data"
CODE15_PARTS = [0]  # one part: ~20k exams, 2.7 GB zipped
N_PER_GROUP = 500   # records per source and class for the shortcut probe
SEED = 42

# Shared Drive folders (link or id) holding the original files; None downloads from Zenodo
CODE15_DRIVE = None
SAMITROP_DRIVE = None

# Processed arrays go to Drive on Colab, so the next notebooks can reuse them
SAVE_TO_DRIVE = IN_COLAB
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROCESSED_DIR = Path("/content/drive/MyDrive/chagas-data/processed")
else:
    PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GROUPS = ["CODE-15% neg", "CODE-15% pos", "SaMi-Trop pos"]
PALETTE = dict(zip(GROUPS, sns.color_palette("colorblind", len(GROUPS))))

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")
N_JOBS = -1  # feature extraction workers

## Data

Same raw files as in `01_dataset_analysis.ipynb`; downloads are skipped when the files already exist.

In [ ]:
download.download_code15(DATA_DIR, CODE15_PARTS, gdown=CODE15_DRIVE)
download.download_samitrop(DATA_DIR, gdown=SAMITROP_DRIVE)
meta = sources.load_all(DATA_DIR, CODE15_PARTS)

## 1. Cohort

Quality checks run once on the raw signals. Dropped records:
- **empty**: entirely zero;
- **short**: shorter than the 7 s window;
- **flat**: a flat stretch longer than 0.5 s on every lead (detached electrodes);
- **duplicate**: an exact copy of an earlier record.

Kept CODE-15% records get 3 repeats of 5 patient-level folds stratified by label. SaMi-Trop records only enter training (fold −1). Ages are capped at 90.

In [ ]:
cohort = pp.build_cohort(meta, seed=SEED)
cohort["group"] = cohort.source + np.where(cohort.chagas, " pos", " neg")
cohort.to_csv(PROCESSED_DIR / "cohort.csv", index=False)
kept = cohort[cohort.array_index >= 0].sort_values("array_index").reset_index(drop=True)

pd.crosstab(cohort.group, cohort["drop"].replace("", "kept")).reindex(GROUPS)

In [ ]:
code15 = kept[kept.source == "CODE-15%"]
(code15.groupby("fold_0").agg(records=("record_id", "size"), patients=("patient_id", "nunique"),
                              positives=("chagas", "sum"), prevalence=("chagas", "mean"))
       .style.format({"prevalence": "{:.2%}"}))

## 2. Filtering and resampling

Per lead: mean removal, 4th-order Butterworth band-pass 0.5–40 Hz applied forward and backward (zero phase), resampling from 400 to 100 Hz, central 7 s. The filter runs on the whole trimmed record, so the window leaves out the edge transients. Only the 8 independent leads are kept.

In [ ]:
examples = kept.groupby("group").sample(1, random_state=SEED).set_index("group").reindex(GROUPS)
fig, axes = plt.subplots(len(GROUPS), 2, figsize=(15, 8), sharex=True)
for row, group, x in zip(axes, GROUPS, sources.read_signals(examples)):
    raw = pp.center_window(ecg.trim_zero_padding(x), sources.FS)
    raw = raw - raw.mean(axis=0)
    clean = pp.preprocess(x)
    for ax, lead in zip(row, ["II", "V1"]):
        ax.plot(np.arange(len(raw)) / sources.FS, raw[:, sources.LEADS.index(lead)], lw=0.7, color="0.65", label="raw, mean removed")
        ax.plot(np.arange(len(clean)) / pp.FS, clean[:, pp.LEADS.index(lead)], lw=0.9, color=PALETTE[group], label="preprocessed")
        ax.set(title=f"{group} · {lead}", ylabel="mV")
axes[0, 0].legend(loc="upper right")
for ax in axes[-1]:
    ax.set_xlabel("s")
plt.tight_layout()

## 3. Processed signals

All kept records, as float32 of shape (records, 700 samples, 8 leads), in the order of `array_index`.

In [ ]:
signals = np.empty((len(kept), int(pp.WINDOW_S * pp.FS), len(pp.LEADS)), dtype=np.float32)
pos = 0
for _, raw in tqdm(sources.iter_signals(kept, chunk_size=2000), total=ceil(len(kept) / 2000)):
    signals[pos:pos + len(raw)] = np.stack([pp.preprocess(x) for x in raw])
    pos += len(raw)
np.save(PROCESSED_DIR / "signals_ml.npy", signals)
signals.shape

## 4. Shortcut probe after preprocessing

The probe of notebook 01, on a sample of the same size, before and after preprocessing. Length and padding are constant after preprocessing and `lead_residual` loses its meaning with 8 leads, so both sides use the amplitude and spectral features only.

In [ ]:
sample = kept.sample(frac=1, random_state=SEED).groupby("group").head(N_PER_GROUP)
COLS = ["flat_run_s", "abs_p99_mv", "abs_median_mv", "line_60hz"] + [f"power_{lo}_{hi}hz" for lo, hi in leakage.BANDS]

raw_feats, clean_feats, raw_psd, clean_psd = [], [], [], []
for chunk, raw in tqdm(sources.iter_signals(sample, chunk_size=500), total=ceil(len(sample) / 500)):
    for i, x in zip(chunk.array_index, raw):
        raw_feats.append(leakage.nuisance_features(x, sources.FS))
        clean_feats.append(leakage.nuisance_features(signals[i], pp.FS))
        raw_psd.append(ecg.psd(ecg.trim_zero_padding(x), sources.FS, leakage.FREQS).mean(axis=1))
        clean_psd.append(ecg.psd(signals[i], pp.FS, leakage.FREQS).mean(axis=1))
raw_feats = pd.DataFrame(raw_feats, index=sample.index)[COLS]
clean_feats = pd.DataFrame(clean_feats, index=sample.index)[COLS].dropna(axis=1, how="all")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
for ax, spectra, title in [(axes[0], raw_psd, "raw"), (axes[1], clean_psd, "preprocessed")]:
    spectra = np.array(spectra)
    for group in GROUPS:
        ax.semilogy(leakage.FREQS, np.nanmedian(spectra[(sample.group == group).to_numpy()], axis=0),
                    color=PALETTE[group], lw=1, label=group)
    ax.set(xlim=(0, 60), title=f"Median PSD, {title}", xlabel="Hz")
axes[0].set_ylabel("PSD (mV²/Hz)")
axes[0].legend()
plt.tight_layout()

In [ ]:
AMPLITUDE = ["abs_median_mv", "abs_p99_mv"]
pd.concat({"raw": raw_feats.groupby(sample.group)[AMPLITUDE].median(),
           "preprocessed": clean_feats.groupby(sample.group)[AMPLITUDE].median()}, axis=1).reindex(GROUPS).round(3)

In [ ]:
g = sample.group
probes = {
    "source | positives (CODE-15% vs SaMi-Trop)": (g != "CODE-15% neg", g == "SaMi-Trop pos"),
    "Chagas | CODE-15% only": (sample.source == "CODE-15%", sample.chagas),
}

results = {}
for name, (mask, target) in probes.items():
    results[name] = {label: roc_auc_score(target[mask], leakage.source_probe(X[mask], target[mask], sample.patient_id[mask], seed=SEED))
                     for label, X in [("raw", raw_feats), ("preprocessed", clean_feats)]}
pd.DataFrame(results).T.round(3)

In [ ]:
mask, target = next(iter(probes.values()))
pd.DataFrame({"raw": leakage.univariate_auc(raw_feats[mask], target[mask]),
              "preprocessed": leakage.univariate_auc(clean_feats[mask], target[mask])}).sort_values("raw", ascending=False).round(3)

## 5. Physiological features

Extracted from the preprocessed signals, following `src/features.py`:
- **rhythm** (lead II R peaks): heart rate, RR coefficient of variation, RMSSD;
- **intervals** (lead II, wavelet delineation): PR, QRS, QTc (Bazett);
- **median beat**: QRS peak-to-peak per lead, frontal axis from I and aVF = II − I/2, terminal QRS forces (late R in V1, late S in I and V6), maximum limb and precordial voltage;
- **ectopic beats**: share of beats with correlation below 0.8 to the median beat;
- **relative band power** per lead in 0.5–5, 5–15 and 15–40 Hz.

Age and sex stay in the cohort table, so models can include or exclude them.

In [ ]:
rows = Parallel(n_jobs=N_JOBS)(delayed(features.ecg_features)(x, pp.FS) for x in tqdm(signals))
feats = pd.DataFrame(rows)
feats.insert(0, "record_id", kept.record_id.to_numpy())
feats.to_csv(PROCESSED_DIR / "features_ml.csv", index=False)
feats = feats.drop(columns="record_id")
print(f"{feats.shape[1]} features, missing values: {feats.isna().mean().mean():.1%}")
feats.isna().mean().sort_values(ascending=False).head(5).map("{:.1%}".format)

Sanity check against the automatic CODE-15% diagnoses: each feature should move in the expected direction.

In [ ]:
is_code15 = (kept.source == "CODE-15%").to_numpy()
checks = {"RBBB": "qrs_ms", "1dAVb": "pr_ms", "SB": "hr_bpm", "ST": "hr_bpm", "AF": "rr_cv"}
rows = {}
for flag, col in checks.items():
    has = kept[flag].astype(str).eq("True").to_numpy() & is_code15
    rows[f"{col} by {flag}"] = {"with": feats.loc[has, col].median(), "without": feats.loc[is_code15 & ~has, col].median(),
                                "records with": has.sum()}
pd.DataFrame(rows).T.round(3)

In [ ]:
KEY = ["hr_bpm", "rr_cv", "pr_ms", "qrs_ms", "qtc_ms", "axis_deg", "v1_terminal_r_mv", "i_terminal_s_mv",
       "limb_max_pp_mv", "precordial_max_pp_mv", "ectopic_frac"]
feats[KEY].groupby(kept.group.to_numpy()).median().reindex(GROUPS).T.round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col, lim in zip(axes, ["qrs_ms", "axis_deg", "v1_terminal_r_mv", "pr_ms"], [(40, 200), (-180, 180), (-0.5, 1.5), (60, 300)]):
    sns.histplot(data=feats.assign(group=kept.group.to_numpy()), x=col, hue="group", hue_order=GROUPS, palette=PALETTE,
                 stat="density", common_norm=False, element="step", bins=50, binrange=lim, ax=ax, legend=ax is axes[0])
plt.tight_layout()

Each feature on its own: ROC AUC for Chagas within CODE-15%, the challenge score obtained by ranking CODE-15% records by that feature, and the AUC for the source among positives (CODE-15% vs SaMi-Trop). AUCs are folded so that 0.5 is uninformative.

In [ ]:
y15 = kept.chagas.to_numpy()[is_code15]
F15 = feats[is_code15].dropna(axis=1, how="all")
positives = kept.chagas.to_numpy()
single = pd.DataFrame({
    "Chagas AUC (CODE-15%)": leakage.univariate_auc(F15, y15),
    "source AUC (positives)": leakage.univariate_auc(feats[positives].dropna(axis=1, how="all"),
                                                    (kept.source[positives] == "SaMi-Trop").to_numpy()),
})
for col in F15:
    v = F15[col].fillna(F15[col].median())
    single.loc[col, "top-5% score (CODE-15%)"] = max(challenge_score(y15, v), challenge_score(y15, -v))
single.sort_values("Chagas AUC (CODE-15%)", ascending=False).head(15).round(3)

## 6. Saved outputs

`cohort.csv` (all records, drop reasons, `array_index`, folds), `signals_ml.npy` and `features_ml.csv`, aligned by `array_index` and `record_id`.

In [ ]:
for path in sorted(PROCESSED_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size / 1e6:,.1f} MB")